# Tier 1 — TextCNN Multi-Column Submission
**SDRF Metadata Prediction | Kaggle: Harmonizing the Data of your Data**

Trains one TextCNN per Tier 1 column (keyword-dominated, closed vocabulary).
All 8 columns finish in ~10 min on Kaggle T4 GPU.

**Tier 1 columns:** `Comment[Instrument]`, `Characteristics[Organism]`, `Characteristics[Sex]`, `Comment[FragmentationMethod]`, `Comment[FragmentMassTolerance]`, `Comment[PrecursorMassTolerance]`, `Comment[FractionIdentifier]`, `Characteristics[BiologicalReplicate]`

In [1]:
from __future__ import annotations

import math
import os
import random
import warnings
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
CUDA available: True
GPU: NVIDIA GeForce GTX 1060


## Configuration
Adjust `TRAIN_PATH` / `TEST_PATH` to match your Kaggle input paths.
On Kaggle, input files are typically under `/kaggle/input/<competition-name>/`.

In [2]:
# ── Paths — update TEST_PATH to your actual test file name ──────────────────
ROOT          = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd().parent
OUTPUT_DIR    = ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH      = OUTPUT_DIR / 'train_preprocessed.csv'
TEST_PATH       = OUTPUT_DIR / 'test_preprocessed.csv'  # ← adjust if needed
SUBMISSION_PATH = OUTPUT_DIR / 'submission_tier1.csv'

# ── Tier 1 columns (TextCNN-suitable) ────────────────────────────────────────
TIER1_COLS = [
    'Comment[Instrument]',
    'Characteristics[Organism]',
    'Characteristics[Sex]',
    'Comment[FragmentationMethod]',
    'Comment[FragmentMassTolerance]',
    'Comment[PrecursorMassTolerance]',
    'Comment[FractionIdentifier]',
    'Characteristics[BiologicalReplicate]',
]

# ── Hyperparameters ───────────────────────────────────────────────────────────
MAX_VOCAB      = 40_000
MIN_FREQ       = 2
MAX_LENGTH     = 512
EMBED_DIM      = 256
NUM_FILTERS    = 128
KERNEL_SIZES   = (3, 5, 7)
DROPOUT        = 0.2
BATCH_SIZE     = 32
LR             = 1e-3
EPOCHS         = 3
RANDOM_STATE   = 42
NOT_APPLICABLE = 'Not Applicable'

def set_seed(seed: int = RANDOM_STATE) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
print('Config loaded.')

Config loaded.


## Vocabulary + Dataset classes
`SimpleVocab` builds a word-level vocabulary from training text.
`CNNDataset` encodes text → fixed-length integer sequences.
`InferenceDataset` does the same without labels (for the test set).

In [3]:
class SimpleVocab:
    PAD, UNK = 0, 1

    def __init__(self, min_freq: int = MIN_FREQ, max_vocab: int = MAX_VOCAB):
        self.min_freq  = min_freq
        self.max_vocab = max_vocab
        self.token_to_id: dict[str, int] = {'<pad>': self.PAD, '<unk>': self.UNK}

    @staticmethod
    def tokenize(text: str) -> list[str]:
        return str(text).lower().split()

    def fit(self, texts: list[str]) -> 'SimpleVocab':
        counter: Counter = Counter()
        for t in texts:
            counter.update(self.tokenize(t))
        for token, freq in counter.most_common(self.max_vocab):
            if freq < self.min_freq or token in self.token_to_id:
                continue
            self.token_to_id[token] = len(self.token_to_id)
        return self

    def encode(self, text: str, max_length: int = MAX_LENGTH) -> list[int]:
        ids = [self.token_to_id.get(t, self.UNK)
               for t in self.tokenize(text)[:max_length]]
        ids += [self.PAD] * (max_length - len(ids))
        return ids

    @property
    def size(self) -> int:
        return len(self.token_to_id)


class CNNDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_length=MAX_LENGTH):
        self.encoded = [vocab.encode(t, max_length) for t in texts]
        self.labels  = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return (torch.tensor(self.encoded[idx], dtype=torch.long),
                torch.tensor(self.labels[idx],  dtype=torch.long))


class InferenceDataset(Dataset):
    def __init__(self, texts, vocab, max_length=MAX_LENGTH):
        self.encoded = [vocab.encode(t, max_length) for t in texts]
    def __len__(self): return len(self.encoded)
    def __getitem__(self, idx):
        return torch.tensor(self.encoded[idx], dtype=torch.long)

print('Dataset classes defined.')

Dataset classes defined.


## TextCNN model
Three parallel Conv1D layers with kernel sizes 3, 5, 7 scan for n-gram features.
Max-over-time pooling collapses each feature map to a single value.
All three pooled vectors are concatenated → dropout → linear classifier.

**Why this works for SDRF:** instrument names, organism names, and method acronyms
appear as short keyword spans. The CNN finds them regardless of position in the document.

In [4]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, num_labels,
                 embed_dim=EMBED_DIM, num_filters=NUM_FILTERS,
                 kernel_sizes=KERNEL_SIZES, dropout=DROPOUT):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, k, padding=k // 2)
            for k in kernel_sizes
        ])
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(num_filters * len(kernel_sizes), num_labels)

    def forward(self, input_ids):
        x = self.embedding(input_ids).transpose(1, 2)       # (B, E, L)
        pooled = [torch.amax(torch.relu(c(x)), dim=2)        # each (B, F)
                  for c in self.convs]
        out = self.dropout(torch.cat(pooled, dim=1))         # (B, 3F)
        return self.classifier(out)                          # (B, num_labels)

print('TextCNN defined.')

TextCNN defined.


## Per-column training + inference
`train_column()` — fits one TextCNN per SDRF column, saves best checkpoint by macro-F1.

`predict_column()` — runs the saved model on test text, maps integer predictions back to label strings.

In [5]:
@dataclass
class ColumnResult:
    col: str
    val_f1: float
    id_to_label: dict
    model: TextCNN
    vocab: SimpleVocab
    label_to_id: dict = field(default_factory=dict)


def train_column(col: str, train_df: pd.DataFrame):
    df = train_df[['text_input', col]].copy()
    df.columns = ['text', 'label']
    df = df[df['label'] != NOT_APPLICABLE].dropna()

    if len(df) < 20:
        print(f'  [SKIP] {col} — only {len(df)} labelled rows')
        return None

    counts = df['label'].value_counts()
    keep   = counts[counts >= 2].index.tolist()
    df     = df[df['label'].isin(keep)].reset_index(drop=True)

    categories     = pd.Categorical(df['label'])
    df['label_id'] = categories.codes
    id_to_label    = dict(enumerate(categories.categories))
    label_to_id    = {v: k for k, v in id_to_label.items()}
    num_labels     = len(id_to_label)

    try:
        tr, va = train_test_split(df, test_size=0.15,
                                  random_state=RANDOM_STATE,
                                  stratify=df['label_id'])
    except ValueError:
        tr, va = train_test_split(df, test_size=0.15, random_state=RANDOM_STATE)

    vocab = SimpleVocab().fit(tr['text'].tolist())
    tr_dl = DataLoader(CNNDataset(tr['text'].tolist(), tr['label_id'].tolist(), vocab),
                       batch_size=BATCH_SIZE, shuffle=True)
    va_dl = DataLoader(CNNDataset(va['text'].tolist(), va['label_id'].tolist(), vocab),
                       batch_size=BATCH_SIZE, shuffle=False)

    model     = TextCNN(vocab.size, num_labels).to(device)
    optimizer = AdamW(model.parameters(), lr=LR)
    loss_fn   = nn.CrossEntropyLoss()
    best_f1, best_state = 0.0, None

    for epoch in range(EPOCHS):
        model.train()
        for ids, labels in tqdm(tr_dl, desc=f'  {col[:30]} ep{epoch+1}', leave=False):
            ids, labels = ids.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(ids), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for ids, labels in va_dl:
                preds.extend(model(ids.to(device)).argmax(1).cpu().tolist())
                targets.extend(labels.tolist())

        f1 = f1_score(targets, preds, average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1    = f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    print(f'  [OK] {col:<45} val macro-F1 = {best_f1:.4f}  classes={num_labels}')
    return ColumnResult(col, best_f1, id_to_label, model, vocab, label_to_id)


def predict_column(result, texts):
    dl = DataLoader(InferenceDataset(texts, result.vocab),
                    batch_size=BATCH_SIZE, shuffle=False)
    result.model.eval().to(device)
    preds = []
    with torch.no_grad():
        for ids in dl:
            preds.extend(result.model(ids.to(device)).argmax(1).cpu().tolist())
    return [result.id_to_label[p] for p in preds]

print('Training and inference functions defined.')

Training and inference functions defined.


## Load & prepare data

In [6]:
print('Loading data …')
train = pd.read_csv(TRAIN_PATH, dtype=str).fillna(NOT_APPLICABLE)

if 'text_input' not in train.columns:
    train['text_input'] = (
        'PXD: ' + train['PXD'].astype(str) + '\n\n'
        + train['pub_text'].astype(str)
          .str.replace(r'\s+', ' ', regex=True)
          .str.strip()
          .str.slice(0, 16_000)
    )

if TEST_PATH.exists():
    test = pd.read_csv(TEST_PATH, dtype=str).fillna('')
    if 'text_input' not in test.columns:
        test['text_input'] = (
            'PXD: ' + test['PXD'].astype(str) + '\n\n'
            + test['pub_text'].astype(str)
              .str.replace(r'\s+', ' ', regex=True)
              .str.strip()
              .str.slice(0, 16_000)
        )
    test_texts = test['text_input'].tolist()
    print(f'Test rows:  {len(test):,}')
else:
    raise FileNotFoundError(
        f"{TEST_PATH} not found. Build test_preprocessed.csv from the Kaggle test set (1659 rows) before submission."
    )

print(f'Train rows: {len(train):,}')
print(f'Columns available: {[c for c in TIER1_COLS if c in train.columns]}')

Loading data …
Test rows:  1,659
Train rows: 36,280
Columns available: ['Comment[Instrument]', 'Characteristics[Organism]', 'Characteristics[Sex]', 'Comment[FragmentationMethod]', 'Comment[FragmentMassTolerance]', 'Comment[PrecursorMassTolerance]', 'Comment[FractionIdentifier]', 'Characteristics[BiologicalReplicate]']


## Label normalisation
The SDRF metadata in this competition was never normalised before labelling, so the
**same physical instrument / method appears under 3–10 different string variants**
in the training set.  Examples for `Comment[Instrument]`:

```
NT=Q Exactive HF;AC=MS:1002523   ← variant A
AC=MS:1002523;NT=Q Exactive HF   ← variant B  (AC/NT swapped)
AC=MS:1002523; NT=Q Exactive HF  ← variant C  (space after semicolon)
Q Exactive HF                    ← variant D  (bare name, no tags)
```

All four refer to the same instrument.  Without normalisation the model learns
~339 spurious classes instead of the ~14 physical instruments, wasting capacity
and making predictions that Kaggle will never score correctly.

**Strategy per column type:**
- `Comment[Instrument]` / `Comment[FragmentationMethod]` — extract the `NT=` name
  as the canonical label (strip `AC=` and whitespace noise entirely).
- `Comment[FragmentMassTolerance]` / `Comment[PrecursorMassTolerance]` — already
  clean numeric+unit strings; just unify `not available` → `Not Applicable`.
- All other Tier-1 columns — unify `not available` / `not applicable` case variants
  to the canonical `Not Applicable`.

In [17]:
import re

# ── NT= extractor ─────────────────────────────────────────────────────────────
# Pulls the human-readable instrument / method name out of NT= tags,
# collapsing all ordering and whitespace variants into a single canonical string.
_NA_VARIANTS = {'not applicable', 'not available', 'n/a', 'na'}

def _extract_nt(s: str) -> str:
    """Return the NT= value if present, otherwise return the stripped bare string."""
    m = re.search(r'\bNT\s*=\s*([^;]+)', s)
    return m.group(1).strip() if m else s.strip()

def normalize_nt_column(series: pd.Series) -> pd.Series:
    """
    For columns where values may be 'NT=X;AC=Y', 'AC=Y;NT=X', 'X', etc.:
    extract only the NT name as the canonical label.
    'Not Applicable' / 'not available' / bare NA variants → NOT_APPLICABLE.
    """
    def _norm(s: str) -> str:
        if pd.isna(s):
            return NOT_APPLICABLE
        s = s.strip()
        if s.lower() in _NA_VARIANTS:
            return NOT_APPLICABLE
        return _extract_nt(s)
    return series.apply(_norm)

def normalize_na_variants(series: pd.Series) -> pd.Series:
    """Unify 'not available' / 'not applicable' case variants → NOT_APPLICABLE."""
    def _norm(s: str) -> str:
        if pd.isna(s):
            return NOT_APPLICABLE
        return NOT_APPLICABLE if s.strip().lower() in _NA_VARIANTS else s.strip()
    return series.apply(_norm)


# ── Apply to train ────────────────────────────────────────────────────────────
_NT_COLS  = ['Comment[Instrument]', 'Comment[FragmentationMethod]']
_NA_COLS  = [c for c in TIER1_COLS if c not in _NT_COLS]

before_counts = {c: train[c].nunique() for c in TIER1_COLS if c in train.columns}

for col in _NT_COLS:
    if col in train.columns:
        train[col] = normalize_nt_column(train[col])
for col in _NA_COLS:
    if col in train.columns:
        train[col] = normalize_na_variants(train[col])

after_counts = {c: train[c].nunique() for c in TIER1_COLS if c in train.columns}

print('Label normalisation — unique class counts before → after:')
for col in TIER1_COLS:
    if col in before_counts:
        b, a = before_counts[col], after_counts[col]
        flag = '  ⬇' if a < b else ''
        print(f'  {col:<45}  {b:>4} → {a:>4}{flag}')

Label normalisation — unique class counts before → after:
  Comment[Instrument]                             339 →   14  ⬇
  Characteristics[Organism]                        19 →   19
  Characteristics[Sex]                              9 →    7  ⬇
  Comment[FragmentationMethod]                     18 →    7  ⬇
  Comment[FragmentMassTolerance]                   26 →   25  ⬇
  Comment[PrecursorMassTolerance]                  14 →   13  ⬇
  Comment[FractionIdentifier]                     118 →  116  ⬇
  Characteristics[BiologicalReplicate]             44 →   42  ⬇


## Train one TextCNN per Tier 1 column
Each column is independent — if one fails the rest still run.
Expected time on Kaggle T4: ~1–2 min per column, ~10 min total.

In [ ]:
results: dict[str, ColumnResult] = {}
available_cols = [c for c in TIER1_COLS if c in train.columns]

for col in available_cols:
    result = train_column(col, train)
    if result is not None:
        results[col] = result

print(f'\nFinished training {len(results)}/{len(available_cols)} columns.')

  Comment[Instrument] ep3:   8%|▊         | 77/964 [00:02<00:29, 30.01it/s] 

## Build & save submission CSV

In [9]:
submission = pd.DataFrame(index=range(len(test)))

if 'ID' in test.columns:
    submission['ID'] = test['ID'].values
elif 'row_id' in test.columns:
    submission['ID'] = test['row_id'].values
else:
    raise ValueError('No ID/row_id column found in test data.')

if 'PXD' in test.columns:
    submission['PXD'] = test['PXD'].values
if 'Raw Data File' in test.columns:
    submission['Raw Data File'] = test['Raw Data File'].values

for col, result in results.items():
    print(f'Predicting {col} …')
    submission[col] = predict_column(result, test_texts)

for col in TIER1_COLS:
    if col not in submission.columns:
        submission[col] = NOT_APPLICABLE

sample_path = ROOT / 'data' / 'SampleSubmission.csv'
if sample_path.exists():
    sample = pd.read_csv(sample_path)
    for col in sample.columns:
        if col not in submission.columns:
            submission[col] = NOT_APPLICABLE
    submission = submission[sample.columns]

submission.to_csv(SUBMISSION_PATH, index=False)
print(f'\nSubmission saved → {SUBMISSION_PATH}')
print(f'Shape: {submission.shape}')
print('Columns:', submission.columns.tolist())
submission.head(3)

Predicting Comment[Instrument] …
Predicting Characteristics[Organism] …
Predicting Characteristics[Sex] …
Predicting Comment[FragmentationMethod] …
Predicting Comment[FragmentMassTolerance] …
Predicting Comment[PrecursorMassTolerance] …
Predicting Comment[FractionIdentifier] …
Predicting Characteristics[BiologicalReplicate] …

Submission saved → c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\outputs\submission_tier1.csv
Shape: (1659, 81)
Columns: ['ID', 'PXD', 'Raw Data File', 'Characteristics[Age]', 'Characteristics[AlkylationReagent]', 'Characteristics[AnatomicSiteTumor]', 'Characteristics[AncestryCategory]', 'Characteristics[BMI]', 'Characteristics[Bait]', 'Characteristics[BiologicalReplicate]', 'Characteristics[CellLine]', 'Characteristics[CellPart]', 'Characteristics[CellType]', 'Characteristics[CleavageAgent]', 'Characteristics[Compound]', 'Characteristics[ConcentrationOfCompound]', 'Characteristics[Depletion]', 'Characteristics[DevelopmentalStage]', '

,ID,PXD,Raw Data File,Characteristics[Age],Characteristics[AlkylationReagent],Characteristics[AnatomicSiteTumor],Characteristics[AncestryCategory],Characteristics[BMI],Characteristics[Bait],Characteristics[BiologicalReplicate],...,FactorValue[Bait],FactorValue[CellPart],FactorValue[Compound],FactorValue[ConcentrationOfCompound].1,FactorValue[Disease],FactorValue[FractionIdentifier],FactorValue[GeneticModification],FactorValue[Temperature],FactorValue[Treatment],Usage
0,1,PXD004010,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable
1,2,PXD004010,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable
2,3,PXD004010,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,1,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable


## Validation summary

In [9]:
import numpy as np

print('Per-column validation macro-F1:')
f1_scores = []
for col, res in results.items():
    print(f'  {col:<45} {res.val_f1:.4f}')
    f1_scores.append(res.val_f1)

if f1_scores:
    print(f'\n  Mean Tier-1 macro-F1: {np.mean(f1_scores):.4f}')
    print(f'  Min:                  {np.min(f1_scores):.4f}')
    print(f'  Max:                  {np.max(f1_scores):.4f}')

Per-column validation macro-F1:
  Comment[Instrument]                           0.9751
  Characteristics[Organism]                     0.5810
  Characteristics[Sex]                          0.6930
  Comment[FragmentationMethod]                  0.7953
  Comment[FragmentMassTolerance]                0.9998
  Comment[PrecursorMassTolerance]               1.0000
  Comment[FractionIdentifier]                   0.0493
  Characteristics[BiologicalReplicate]          0.0990

  Mean Tier-1 macro-F1: 0.6491
  Min:                  0.0493
  Max:                  1.0000


## Rule-based extractors: `Comment[FractionIdentifier]` & `Characteristics[BiologicalReplicate]`

Both columns contain integer row identifiers that TextCNN cannot reliably learn
because they are **positional** (which fraction / replicate is this row) rather
than semantic.  The rule-based approach:

1. **Parse pub_text** for method-specific language ("24 SCX fractions", "in biological
   triplicate", "3 biological replicates") using regex patterns that avoid false
   positives ("volume fraction", "collagen fraction", "P100 fraction", etc.).
2. **Extract N** — the dimensionality of that axis for the whole dataset.
3. **Assign 1 … N cyclically** across the rows that share the same PXD, capped so
   we never emit a value beyond row count.
4. **Fall back to `Not Applicable`** when no reliable integer is found.

In [10]:
import re
from collections import Counter

# ── Patterns that indicate LC/gel sample fractionation with a count ─────────
# Narrowly scoped to avoid false positives from "collagen volume fraction",
# "P100 fraction", "nuclear fraction", etc.
_FRAC_PATTERNS = [
    # "24 SCX fractions", "18 high-pH fractions", "8 bRP fractions", etc.
    r'\b(\d{1,3})\s+(?:SCX|SAX|IEF|bRP|RPLC|HpRP|RP|gel|iTRAQ|strong[- ]cation|high[- ]pH)\s+fraction[s]?\b',
    # "fractionated into 24 [fractions]", "fractionation into 18 parts"
    r'\bfractionat\w+\s+into\s+(\d{1,3})\b',
    # "24-fraction", "an 18-fraction pre-fractionation"
    r'\b(\d{1,3})\s*[-–]\s*fraction\b',
    # "separated / divided / split into 24 fractions"
    r'\b(?:separat|divid|split|cut)\w*\s+into\s+(\d{1,3})\s+fraction[s]?\b',
    # "24 fractions were collected / pooled / analyzed"
    r'\b(\d{1,3})\s+fraction[s]?\s+(?:were|was)\s+(?:collect|pool|analyz|generat|used)',
]

# ── Patterns for biological replicate count ──────────────────────────────────
_REP_WORDS = {
    'quadruplicate': 4, 'quadruplicates': 4,
    'triplicate':    3, 'triplicates':    3,
    'duplicate':     2, 'duplicates':     2,
}
_REP_PATTERNS = [
    r'\b(\d{1,2})\s+(?:independent\s+)?biological\s+replicate[s]?\b',
    r'\b(\d{1,2})\s+replicate[s]?\s+(?:per\s+(?:sample|condition|experiment)|each\b)',
    r'\b(?:run|analyz\w+|measur\w+|perform\w+|conduct\w+)\s+(?:in\s+)?(\d{1,2})\s+(?:biological\s+)?replicate[s]?\b',
]


def extract_frac_count(text: str) -> int | None:
    """
    Return the most-frequently mentioned LC/gel fraction count in pub_text,
    restricted to method-context phrases.  Returns None when no signal found.
    """
    candidates = []
    for p in _FRAC_PATTERNS:
        for m in re.finditer(p, text, re.IGNORECASE):
            try:
                n = int(m.group(1))
                if 2 <= n <= 200:
                    candidates.append(n)
            except (IndexError, ValueError):
                pass
    return Counter(candidates).most_common(1)[0][0] if candidates else None


def extract_rep_count(text: str) -> int | None:
    """
    Return the biological replicate count inferred from pub_text.
    Keyword synonyms (triplicate → 3, duplicate → 2) take priority.
    Returns None when no signal found.
    """
    # Keyword synonyms — check in a sentence that contains a replicate-related word
    for token, n in _REP_WORDS.items():
        if re.search(
            r'\b(?:biological\s+' + token
            + r'|in\s+(?:biological\s+)?' + token
            + r'|' + token + r'\s+(?:experiment|replicate|analysis|run|assay|measurement))\b',
            text, re.IGNORECASE
        ):
            return n
        # Plain occurrence (less strict)
        if re.search(r'\b' + token + r'\b', text, re.IGNORECASE):
            return n

    candidates = []
    for p in _REP_PATTERNS:
        for m in re.finditer(p, text, re.IGNORECASE):
            try:
                n = int(m.group(1))
                if 2 <= n <= 20:
                    candidates.append(n)
            except (IndexError, ValueError):
                pass
    return Counter(candidates).most_common(1)[0][0] if candidates else None


def assign_cyclic(n_rows: int, n_vals: int | None) -> list[str]:
    """
    Assign 1-indexed labels cycling through 1 … min(n_vals, n_rows).
    Returns ['Not Applicable'] * n_rows when n_vals is None.
    """
    if n_vals is None:
        return [NOT_APPLICABLE] * n_rows
    effective = min(n_vals, n_rows)
    return [str((i % effective) + 1) for i in range(n_rows)]


# ── Probe: show what each test PXD extracts ──────────────────────────────────
print(f"{'PXD':<15} {'rows':>5}  {'n_fracs':>8}  {'n_reps':>7}")
print('─' * 42)
for pxd, grp in test.groupby('PXD'):
    text = grp['pub_text'].iloc[0]
    print(f"{pxd:<15} {len(grp):>5}  {str(extract_frac_count(text)):>8}  {str(extract_rep_count(text)):>7}")

PXD              rows   n_fracs   n_reps
──────────────────────────────────────────
PXD004010          10      None     None
PXD016436          18      None        3
PXD019519           6      None     None
PXD025663          12      None     None
PXD040582          24      None        3
PXD050621           9      None        3
PXD061009           2      None        3
PXD061090           6      None        2
PXD061136           2      None        3
PXD061195        1376      None       10
PXD061285          60      None        3
PXD062014          24      None        3
PXD062469          32      None        3
PXD062877          48      None     None
PXD064564          30      None     None


In [12]:
# ── Reload saved submission, patch the two weak columns, re-save ─────────────
sub2 = pd.read_csv(SUBMISSION_PATH, dtype=str)

for pxd, grp in test.groupby('PXD'):
    text  = grp['pub_text'].iloc[0]
    idx   = grp.index                       # same positional index as sub2 row

    n_f = extract_frac_count(text)
    n_r = extract_rep_count(text)

    sub2.loc[idx, 'Comment[FractionIdentifier]']          = assign_cyclic(len(idx), n_f)
    sub2.loc[idx, 'Characteristics[BiologicalReplicate]'] = assign_cyclic(len(idx), n_r)

sub2.to_csv(SUBMISSION_PATH, index=False)
print(f'Updated submission saved → {SUBMISSION_PATH}')
print(f'Shape: {sub2.shape}')

# ── Spot-check ────────────────────────────────────────────────────────────────
print('\nComment[FractionIdentifier] value counts:')
print(sub2['Comment[FractionIdentifier]'].value_counts().head(10).to_string())
print('\nCharacteristics[BiologicalReplicate] value counts:')
print(sub2['Characteristics[BiologicalReplicate]'].value_counts().head(10).to_string())

Updated submission saved → c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\outputs\submission_tier1.csv
Shape: (1659, 81)

Comment[FractionIdentifier] value counts:
Comment[FractionIdentifier]
Not Applicable    1659

Characteristics[BiologicalReplicate] value counts:
Characteristics[BiologicalReplicate]
1     199
2     199
3     193
4     138
5     138
6     138
7     137
8     137
9     137
10    137


In [14]:
# Resolve paths using notebook config when available
outputs_dir = OUTPUT_DIR if 'OUTPUT_DIR' in globals() else Path('outputs')
sub_path = SUBMISSION_PATH if 'SUBMISSION_PATH' in globals() else outputs_dir / 'submission_tier1.csv'
sample_path = (ROOT / 'data' / 'SampleSubmission.csv') if 'ROOT' in globals() else Path('data') / 'SampleSubmission.csv'

# Fallback: pick any submission*.csv in outputs if exact file is missing
if not Path(sub_path).exists():
	candidates = sorted(Path(outputs_dir).glob('submission*.csv'))
	if candidates:
		sub_path = candidates[-1]

sub = pd.read_csv(sub_path, dtype=str) if Path(sub_path).exists() else pd.DataFrame()
sample = pd.read_csv(sample_path, dtype=str) if Path(sample_path).exists() else pd.DataFrame()

if sub.empty:
	print(f"Submission file not found. Expected: {sub_path}")
else:
	print(f"Using submission file: {sub_path}")
if sample.empty:
	print(f"Sample submission file not found. Expected: {sample_path}")

if not sub.empty:
	print("Your columns:", sub.columns[:5].tolist())
if not sample.empty:
	print("Sample columns:", sample.columns[:5].tolist())

print()
print("Your Comment[Instrument] top 5 values:")
if not sub.empty and 'Comment[Instrument]' in sub.columns:
	print(sub['Comment[Instrument]'].value_counts().head())
else:
	print("Column Comment[Instrument] not found in submission.")

print()
print("Train Comment[Instrument] top 5 values:")
train_df = train if 'train' in globals() else pd.read_csv(
	TRAIN_PATH if 'TRAIN_PATH' in globals() else outputs_dir / 'train_preprocessed.csv',
	dtype=str
)
if 'Comment[Instrument]' in train_df.columns:
	print(train_df['Comment[Instrument]'].value_counts().head())
else:
	print("Column Comment[Instrument] not found in train data.")

Using submission file: c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\outputs\submission_tier1.csv
Your columns: ['ID', 'PXD', 'Raw Data File', 'Characteristics[Age]', 'Characteristics[AlkylationReagent]']
Sample columns: ['ID', 'PXD', 'Raw Data File', 'Characteristics[Age]', 'Characteristics[AlkylationReagent]']

Your Comment[Instrument] top 5 values:
Comment[Instrument]
AC=MS:1002523;NT=Q Exactive HF         1597
AC=MS:1001742;NT=LTQ Orbitrap Velos      60
NT=Q Exactive HF;AC=MS:1002523            2
Name: count, dtype: int64

Train Comment[Instrument] top 5 values:
Comment[Instrument]
NT=Q Exactive;AC=MS:1001911            4855
NT=Q Exactive Plus;AC=MS:1001911       4424
AC=MS:1001742;NT=LTQ Orbitrap Velos    3976
AC=MS:1002523;NT=Q Exactive HF         2994
AC=MS:1001910;NT=LTQ Orbitrap Elite    2938
Name: count, dtype: int64


In [16]:

# Reuse already loaded dataframe when available (avoids path issues)
if 'train' in globals() and isinstance(train, pd.DataFrame):
	train_src = train
elif 'train_df' in globals() and isinstance(train_df, pd.DataFrame):
	train_src = train_df
else:
	candidate_paths = []
	if 'TRAIN_PATH' in globals():
		candidate_paths.append(Path(TRAIN_PATH))
	if 'OUTPUT_DIR' in globals():
		candidate_paths.append(Path(OUTPUT_DIR) / 'train_preprocessed.csv')
	candidate_paths.extend([
		Path.cwd() / 'outputs' / 'train_preprocessed.csv',
		Path('outputs') / 'train_preprocessed.csv',
	])

	found_path = next((p for p in candidate_paths if p.exists()), None)
	train_src = pd.read_csv(found_path, dtype=str) if found_path is not None else pd.DataFrame()

# What are the EXACT strings Kaggle expects for this instrument?
if train_src.empty:
	print("train_preprocessed.csv not found, and no in-memory train dataframe is available.")
elif 'Comment[Instrument]' not in train_src.columns:
	print("Column 'Comment[Instrument]' not found in train data.")
else:
	print(train_src['Comment[Instrument]'].value_counts(dropna=False).head(20).to_string())

Comment[Instrument]
NT=Q Exactive;AC=MS:1001911               4855
NT=Q Exactive Plus;AC=MS:1001911          4424
AC=MS:1001742;NT=LTQ Orbitrap Velos       3976
AC=MS:1002523;NT=Q Exactive HF            2994
AC=MS:1001910;NT=LTQ Orbitrap Elite       2938
AC=MS:1002634;NT=Q Exactive Plus          1926
AC=MS:1000639;NT=Orbitrap Fusion          1687
NT=Q Exactive HF;AC=MS:1002523            1587
NT=Orbitrap Fusion Lumos;AC=MS:1002731    1460
Q Exactive Plus                           1346
AC=MS:1002732;NT=Orbitrap Fusion Lumos    1318
AC=MS:1002523; NT=Q Exactive HF           1290
AC=MS:1001911;NT=Q Exactive               1148
NT=Q Exactive Plus;AC=MS:1002634          1107
NT=LTQ Orbitrap;AC=MS:1000449             1016
LTQ Orbitrap XL                            528
NT=Orbitrap Fusion Lumos                   480
NT=Orbitrap Fusion Lumos;AC=MS:1002732     431
NT=LTQ Orbitrap Velos;AC=MS:1001742        351
NT=LTQ Orbitrap Elite;AC=MS:1001910        246
